# Identificação do projeto

**Título do projeto:** Perfil das faixas populares nos charts da Deezer por gênero
**Integrantes do grupo:** Eduardo Fabri, João Correa, Rodrigo Alves e João Pedro Cardoso
**Descrição breve do problema ou contexto:** os charts da Deezer dizem quais faixas estão no topo de cada gênero, mas não o que elas têm em comum. O projeto coleta esses charts pela API pública da Deezer e prepara os dados para comparar, entre gêneros, a duração das faixas, a idade do lançamento, a popularidade do artista e a concentração de artistas no ranking.

Preencha este notebook ao longo dos dois RAs. Registre decisões e interpretações em poucas frases e acrescente as células de código necessárias em cada etapa.

## Preparação do ambiente

Esta célula prepara as bibliotecas, as pastas do projeto e a origem dos dados. No Google Colab, ela clona o repositório para ter acesso ao retrato bruto da API já coletado.

In [ ]:
# Preparação do ambiente: bibliotecas, pastas do projeto e origem dos dados
import subprocess
import sys
from pathlib import Path

import pandas as pd

URL_REPOSITORIO = "https://github.com/Jcliz/data-analytics.git"
RAMO = "main"  # no Colab, troque pela branch da tarefa para testar antes do merge (ex.: "t02-coleta")
NO_COLAB = "google.colab" in sys.modules

if NO_COLAB:
    if Path("data-analytics").exists():
        # Clone de uma execução anterior nesta sessão do Colab: traz a versão mais nova do ramo
        subprocess.run(["git", "-C", "data-analytics", "pull", "-q", "origin", RAMO], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "-b", RAMO, URL_REPOSITORIO], check=True)

# A raiz é a pasta que contém o CONTEXT.md (funciona no Colab, na raiz ou dentro de notebook/)
candidatos = [Path("data-analytics"), Path.cwd(), Path.cwd().parent]
RAIZ = next((p.resolve() for p in candidatos if (p / "CONTEXT.md").exists()), None)
if RAIZ is None:
    raise FileNotFoundError(
        "Raiz do repositório não encontrada (pasta com o CONTEXT.md). Rode o notebook de dentro do clone "
        f"(raiz ou notebook/) ou, no Colab, confira se o ramo '{RAMO}' já tem a estrutura do projeto."
    )

PASTA_BRUTOS = RAIZ / "dados" / "brutos"
PASTA_PREPARADOS = RAIZ / "dados" / "preparados"
PASTA_STREAMING = RAIZ / "dados" / "streaming"
for pasta in (PASTA_BRUTOS, PASTA_PREPARADOS, PASTA_STREAMING):
    pasta.mkdir(parents=True, exist_ok=True)

# False: lê o retrato bruto versionado (ADR-0004). True: coleta de novo na API da Deezer.
BAIXAR_DA_API = False

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
print(f"Raiz do projeto: {RAIZ}")
print(f"Ambiente: {'Google Colab' if NO_COLAB else 'local'} · Python {sys.version.split()[0]} · pandas {pd.__version__}")

---

# RA1 — Data Ingestion, Cleaning and Initial Analysis

Nesta etapa, carregue, prepare e explore os dados do projeto.

## 1. Contexto e objetivo

Apresente brevemente o problema investigado e o que o grupo pretende descobrir com os dados.

**Resposta do grupo:**

Os charts da Deezer mostram as faixas mais ouvidas em cada gênero, mas não o que elas têm em comum. O grupo quer descrever o perfil dessas faixas e ver se ele muda entre gêneros. O recorte é um único retrato dos charts da Deezer, o chart geral e o de cada gênero, na data da coleta registrada na seção 2. A unidade de observação é a faixa. As perguntas são:

- **P1:** como a duração das faixas varia entre os gêneros;
- **P2:** que parte das faixas de cada gênero foi lançada nos 12 meses anteriores à coleta;
- **P3:** se faixas de artistas com mais fãs têm `rank` mais alto, no geral e dentro de cada gênero;
- **P4:** quantos artistas distintos ocupam cada chart de gênero e que fatia das posições fica com os 5 artistas com mais faixas.

O detalhe do recorte e das lacunas está na seção 01 do documento do RA1.

## 2. Fonte e ingestão dos dados

- Identifique a fonte dos dados, incluindo link ou referência de acesso.
- Descreva brevemente o dataset e o que seus registros representam.
- Explique como os dados foram carregados e implemente a ingestão.

**Resposta do grupo:**

[Preencha aqui.]

## 3. Exploração inicial dos dados

Investigue a quantidade de registros e colunas, os tipos de dados, os valores ausentes e outras características relevantes da base.

Uma linha representa uma ocorrência de faixa em um chart no retrato da coleta.
A chave candidata é o par `chart_id_genero` e `id`, dentro desse retrato.
`id` identifica a faixa, mas não é único entre charts. As verificações abaixo
contam ocorrências, não faixas distintas, salvo indicação explícita.

In [ ]:
# Visão geral e tipos observados
import json
import numpy as np
import pandas as pd
from IPython.display import display

numero_linhas = len(df_bruto)
if numero_linhas == 0:
    raise ValueError("O diagnóstico exige um df_bruto não vazio; execute a seção 2.")
print(f"Retrato: {numero_linhas} linhas × {df_bruto.shape[1]} colunas; "
      f"{df_bruto['id'].nunique()} faixas; {df_bruto['chart_id_genero'].nunique()} charts.")
print(f"Memória, incluindo objetos: {df_bruto.memory_usage(deep=True).sum():,} bytes.")
inventario = df_bruto.dtypes.astype(str).rename("tipo_observado").to_frame()
display(inventario.reset_index(names="atributo"))

In [ ]:
# Ausências nulas e textos vazios
# Listas e dicionários são objetos válidos: não são tratados como texto.
def texto_vazio(valor):
    return isinstance(valor, str) and not valor.strip()

mascara_ausencias = df_bruto.isna() | df_bruto.apply(lambda coluna: coluna.map(texto_vazio))
ausencias = pd.DataFrame({
    "nulos": df_bruto.isna().sum(),
    "textos_vazios": df_bruto.apply(lambda coluna: coluna.map(texto_vazio)).sum(),
    "total_ausentes": mascara_ausencias.sum(),
})
ausencias["percentual"] = 100 * ausencias["total_ausentes"] / numero_linhas
display(ausencias.reset_index(names="atributo"))

In [ ]:
# Duplicidade e limite da chave
# Serialização canônica permite comparar linhas com listas/dicionários, sem modificar o bruto.
linhas_canonicas = df_bruto.apply(
    lambda coluna: coluna.map(lambda valor: json.dumps(valor, ensure_ascii=False, sort_keys=True)
                             if isinstance(valor, (list, dict)) else valor)
)
duplicadas_inteiras = linhas_canonicas.duplicated()
duplicadas_id = df_bruto.duplicated("id")
duplicadas_chart = df_bruto.duplicated(["chart_id_genero", "id"])
print(f"Ocorrências excedentes por id: {duplicadas_id.sum()}/{numero_linhas}; "
      f"por par chart-faixa: {duplicadas_chart.sum()}/{numero_linhas}; "
      f"linhas inteiras iguais: {duplicadas_inteiras.sum()}/{numero_linhas}.")

In [ ]:
# Estatísticas descritivas e domínios numéricos
colunas_numericas = ["duration", "rank", "position", "detalhe_faixa_bpm",
                    "detalhe_faixa_gain", "detalhe_artista_nb_fan"]
numericos = df_bruto[colunas_numericas].apply(pd.to_numeric, errors="coerce")
display(numericos.describe().T)
# São regras de integridade, não indicadores analíticos das perguntas P1 a P4.
colunas_inteiras = ["id", "chart_id_genero", "chart_ordem", "duration", "rank", "position",
                   "artist_id", "album_id", "detalhe_faixa_id", "detalhe_album_id",
                   "detalhe_artista_id", "detalhe_artista_nb_fan"]
tipos_invalidos = pd.Series(False, index=df_bruto.index)
for coluna in colunas_inteiras:
    valores = pd.to_numeric(df_bruto[coluna], errors="coerce")
    tipos_invalidos |= valores.isna() | ~np.isfinite(valores) | valores.mod(1).ne(0)
# Strings conversíveis também são divergências do tipo esperado, mesmo sem perda na conversão.
for coluna in colunas_inteiras + ["detalhe_faixa_bpm", "detalhe_faixa_gain"]:
    tipos_invalidos |= ~df_bruto[coluna].map(
        lambda valor: isinstance(valor, (int, float, np.integer, np.floating))
        and not isinstance(valor, (bool, np.bool_))
    )
tipos_invalidos |= ~df_bruto["explicit_lyrics"].map(lambda valor: isinstance(valor, (bool, np.bool_)))
invalidos_numericos = (
    numericos.isna().any(axis=1) | ~np.isfinite(numericos).all(axis=1)
    | numericos["duration"].le(0) | numericos["rank"].lt(0)
    | numericos["position"].le(0) | numericos["detalhe_artista_nb_fan"].lt(0)
    | numericos["detalhe_faixa_bpm"].lt(0)
)
for coluna in ["id", "artist_id", "album_id"]:
    invalidos_numericos |= pd.to_numeric(df_bruto[coluna], errors="coerce").le(0)
bpm_zero = numericos["detalhe_faixa_bpm"].eq(0)
gain_zero = numericos["detalhe_faixa_gain"].eq(0)
# gain é um ajuste em dB: negativo é permitido; zero é sinalizado para avaliação, não prova de erro.

In [ ]:
# Datas, formatos e categorias
colunas_datas = ["detalhe_faixa_release_date", "detalhe_album_release_date"]
datas = df_bruto[colunas_datas].apply(lambda coluna: pd.to_datetime(coluna, format="%Y-%m-%d", errors="coerce"))
formatos_data = df_bruto[colunas_datas].apply(
    lambda coluna: coluna.astype("string").str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False)
)
data_coleta = pd.Timestamp(metadados["inicio_utc"]).tz_convert("UTC").tz_localize(None).normalize()
datas_invalidas = (~formatos_data | datas.isna() | datas.lt(pd.Timestamp("1900-01-01"))
                   | datas.gt(data_coleta)).any(axis=1)
textos = df_bruto[["title", "artist_name", "chart_nome_genero"]].astype("string")
espacos = textos.ne(textos.apply(lambda coluna: coluna.str.strip())).any(axis=1)
nomes_por_artista = df_bruto.groupby("artist_id")["artist_name"].nunique()
variacoes_artista = df_bruto["artist_id"].isin(nomes_por_artista[nomes_por_artista.gt(1)].index)
nomes_por_genero = df_bruto.groupby("chart_id_genero")["chart_nome_genero"].nunique()
categorias_inconsistentes = df_bruto["chart_id_genero"].isin(nomes_por_genero[nomes_por_genero.gt(1)].index)
nomes_normalizados = df_bruto["chart_nome_genero"].astype("string").str.strip().str.casefold()
ids_por_nome = df_bruto.groupby(nomes_normalizados)["chart_id_genero"].nunique()
categorias_inconsistentes |= nomes_normalizados.isin(ids_por_nome[ids_por_nome.gt(1)].index)
chart_geral = df_bruto["chart_id_genero"].eq(0)
genero_album_generico = pd.to_numeric(df_bruto["detalhe_album_genre_id"], errors="coerce").le(0)

In [ ]:
# Possíveis outliers e integridade do enriquecimento
# Limites IQR são sinais para inspeção; extremos não são removidos.
limites_outliers = []
for coluna in ["duration", "rank"]:
    primeiro_quartil, terceiro_quartil = numericos[coluna].quantile([0.25, 0.75])
    amplitude = terceiro_quartil - primeiro_quartil
    inferior, superior = primeiro_quartil - 1.5 * amplitude, terceiro_quartil + 1.5 * amplitude
    mascara = numericos[coluna].lt(inferior) | numericos[coluna].gt(superior)
    limites_outliers.append({"atributo": coluna, "inferior": inferior, "superior": superior,
                             "ocorrencias": int(mascara.sum()), "denominador": numero_linhas})
    if coluna == "duration":
        outliers_duracao = mascara
    else:
        outliers_rank = mascara
display(pd.DataFrame(limites_outliers))
display(df_bruto.loc[outliers_duracao, ["id", "title", "duration", "chart_nome_genero"]])
integridade = pd.Series(False, index=df_bruto.index)
for origem, detalhe in [("id", "detalhe_faixa_id"), ("album_id", "detalhe_album_id"),
                        ("artist_id", "detalhe_artista_id")]:
    integridade |= df_bruto[origem].isna() | df_bruto[detalhe].isna() | df_bruto[origem].ne(df_bruto[detalhe])
divergencia_duracao = df_bruto["duration"].ne(df_bruto["detalhe_faixa_duration"])
divergencia_rank = df_bruto["rank"].ne(df_bruto["detalhe_faixa_rank"])

In [ ]:
# Diagnóstico de qualidade Q1 a Q21
campos_essenciais = ["id", "chart_id_genero", "chart_nome_genero", "title", "duration", "rank",
                    "position", "artist_id", "artist_name", "album_id", "detalhe_faixa_release_date",
                    "detalhe_artista_nb_fan"]
verificacoes = [
    ("Q1", "Ausência em pelo menos um campo essencial às P1–P4 ou RA2", mascara_ausencias[campos_essenciais].any(axis=1), "Ausências nulas e textos vazios"),
    ("Q2", "Ausência de preview (nulo ou texto vazio)", mascara_ausencias["preview"], "Ausências nulas e textos vazios"),
    ("Q3", "Ausência de title_version (nulo ou texto vazio)", mascara_ausencias["title_version"], "Ausências nulas e textos vazios"),
    ("Q4", "Ocorrências excedentes por id, após a primeira", duplicadas_id, "Duplicidade e limite da chave"),
    ("Q5", "Ocorrências excedentes pelo par chart_id_genero e id", duplicadas_chart, "Duplicidade e limite da chave"),
    ("Q6", "Linhas inteiras repetidas, após a primeira", duplicadas_inteiras, "Duplicidade e limite da chave"),
    ("Q7", "Tipo divergente de numérico inteiro/real ou booleano esperado", tipos_invalidos, "Estatísticas descritivas e domínios numéricos"),
    ("Q8", "Numéricos não finitos/ausentes ou fora dos domínios descritos", invalidos_numericos, "Estatísticas descritivas e domínios numéricos"),
    ("Q9", "bpm igual a zero (sinal de dado não informado)", bpm_zero, "Estatísticas descritivas e domínios numéricos"),
    ("Q10", "gain igual a zero (a avaliar, não erro comprovado)", gain_zero, "Estatísticas descritivas e domínios numéricos"),
    ("Q11", "Data fora de AAAA-MM-DD, inválida ou fora de 1900 até a coleta", datas_invalidas, "Datas, formatos e categorias"),
    ("Q12", "Espaços nas pontas de título, artista ou gênero", espacos, "Datas, formatos e categorias"),
    ("Q13", "Mais de um nome para um id de artista", variacoes_artista, "Datas, formatos e categorias"),
    ("Q14", "Gênero com mais de um nome ou nome normalizado com mais de um id", categorias_inconsistentes, "Datas, formatos e categorias"),
    ("Q15", "Chart geral id 0: não representa um gênero específico", chart_geral, "Datas, formatos e categorias"),
    ("Q16", "Gênero do álbum genérico (id menor ou igual a zero)", genero_album_generico, "Datas, formatos e categorias"),
    ("Q17", "Duração fora de Q1 − 1,5 IQR a Q3 + 1,5 IQR", outliers_duracao, "Possíveis outliers e integridade do enriquecimento"),
    ("Q18", "Rank fora de Q1 − 1,5 IQR a Q3 + 1,5 IQR", outliers_rank, "Possíveis outliers e integridade do enriquecimento"),
    ("Q19", "Id ausente ou divergente na junção de faixa/álbum/artista", integridade, "Possíveis outliers e integridade do enriquecimento"),
    ("Q20", "Duração do chart diferente da duração em /track", divergencia_duracao, "Possíveis outliers e integridade do enriquecimento"),
    ("Q21", "Rank do chart diferente do rank em /track", divergencia_rank, "Possíveis outliers e integridade do enriquecimento"),
]
diagnostico = pd.DataFrame([
    {"ID": identificador, "regra": regra, "quantidade": int(mascara.sum()),
     "denominador": numero_linhas, "percentual": 100 * mascara.sum() / numero_linhas,
     "celula": celula}
    for identificador, regra, mascara, celula in verificacoes
])
display(diagnostico)
print("As proporções usam ocorrências nos charts e podem se sobrepor; não devem ser somadas.")

**Resposta do grupo:**

No retrato de 16/10/2026, são 2.800 ocorrências, 102 colunas e 2.581 faixas.
Há 219/2.800 repetições excedentes por `id` (7,82%), mas nenhuma pela chave
chart-faixa. BPM é zero em 1.890/2.800 ocorrências (67,50%); `title_version`
está ausente em 2.533/2.800 (90,46%). Uma ocorrência tem lançamento posterior
à coleta, e 143/2.800 (5,11%) têm duração fora dos limites IQR.
Evidências: células “Visão geral e tipos observados” e “Diagnóstico de qualidade Q1 a Q21”.
A repetição por `id` exige preservar o chart de origem
na preparação, pois ele define os grupos das perguntas P1 a P4. O chart geral deve
ser identificado separadamente dos gêneros. BPM zero e versões sem texto precisam
de interpretação; valores extremos de duração são candidatos à inspeção, sem
exclusão automática. O diagnóstico completo e as prioridades estão no documento
“Compreensão e qualidade”, com as mesmas contagens do quadro Q1 a Q21.

## 4. Limpeza e preparação dos dados

Realize a limpeza necessária ao projeto. Verifique valores ausentes, duplicados e inconsistências e trate os problemas encontrados, quando necessário.

**Resposta do grupo:**

[Indique os problemas encontrados e como foram tratados. Se não houve necessidade de tratamento, registre isso brevemente.]

## 5. Transformação dos dados

Realize as transformações necessárias à análise, como criação de colunas, filtros, agrupamentos, conversões ou junções, quando aplicáveis.

**Resposta do grupo:**

[Explique brevemente as transformações realizadas e sua finalidade. Se nenhuma foi necessária, registre isso.]

## 7. Considerações do RA1

Registre brevemente:

- O que o grupo descobriu sobre os dados.
- Quais dificuldades ou limitações encontrou.
- Como os dados preparados poderão ser utilizados nas próximas etapas.

**Resposta do grupo:**

[Preencha aqui.]

---

# RA2 — Streaming Data Processing and Time Windows

Utilize os dados e o contexto do projeto para demonstrar o processamento em streaming e a aplicação de janelas temporais. Escolha um dos cenários:

- **Cenário A — Dados estáticos:** simule a chegada de novos registros utilizando dados artificiais coerentes com o contexto do projeto.
- **Cenário B — Dados em streaming:** utilize sua própria fonte de dados contínuos, sem necessidade de simulação.

## 8. Objetivo da análise em streaming

Responda brevemente:

- O que o grupo deseja monitorar ou analisar?
- Qual informação espera obter com o processamento em streaming?

**Resposta do grupo:**

[Preencha aqui.]

## 9. Fonte de dados e processamento

- Informe o cenário escolhido: dados reais em streaming ou simulação.
- Explique brevemente como os novos registros chegam.
- Implemente o processamento em tempo real.
- Demonstre o recebimento de novos registros.

**Resposta do grupo:**

[Preencha aqui.]

## 10. Aplicação de janelas temporais

- Utilize uma janela **Tumbling** (intervalos sem sobreposição) ou **Sliding** (intervalos que podem se sobrepor).
- Informe o tipo e a duração da janela e o deslocamento, quando aplicável.
- Implemente uma agregação coerente com a análise, como contagem, soma ou média.
- Explique brevemente o que será calculado em cada intervalo.

O foco é demonstrar a compreensão do conceito; não é necessário justificar tecnicamente a duração escolhida.

**Resposta do grupo:**

[Informe o tipo, a duração, o deslocamento (quando aplicável) e o que será calculado em cada janela.]

## 11. Resultados e interpretação

Apresente os resultados do processamento e demonstre como a análise se comporta com a chegada de novos registros.

**Resposta do grupo:**

[Explique brevemente o significado dos resultados e as mudanças observadas com a chegada de novos registros.]

## 12. Considerações do RA2

Responda brevemente:

- O que foi possível observar com o processamento em streaming?
- Qual foi a contribuição das janelas temporais para a análise?
- Essa abordagem faria sentido em uma aplicação real do projeto? Por quê?

Na última resposta, o grupo pode reconhecer que o streaming foi utilizado apenas como exercício didático, se esse for o caso.

**Resposta do grupo:**

[Preencha aqui.]